In [1]:
! pip install scikit-learn

In [2]:
from pymongo import MongoClient
import pandas as pd
from sklearn.linear_model import LinearRegression # 기본적인 리니어 리그레션(선형회귀모델) 예측모델 활용

In [11]:
## MongoDB 연결
client = MongoClient("mongodb://localhost:27017/")
db = client["ai_test"]

## 신규 컬렉션 생성
train_collection = db["train_data"]
predict_collection = db["prediction_result"]

In [12]:
## 샘플 데이터 저장
sample_data = [
    {"temperature": 70, "pressure":100, "":0.10, "quality_score":80},
    {"temperature": 75, "pressure":110, "vibration":0.15, "quality_score":85},
    {"temperature": 80, "pressure":120, "vibration":0.20, "quality_score":88},
    {"temperature": 85, "pressure":130, "vibration":0.25, "quality_score":92},
    {"temperature": 90, "pressure":140, "vibration":0.30, "quality_score":95},
]
train_collection.delete_many({})
train_collection.insert_many(sample_data)

InsertManyResult([ObjectId('6a1d19bd7d6c50bc711d1cb8'), ObjectId('6a1d19bd7d6c50bc711d1cb9'), ObjectId('6a1d19bd7d6c50bc711d1cba'), ObjectId('6a1d19bd7d6c50bc711d1cbb'), ObjectId('6a1d19bd7d6c50bc711d1cbc')], acknowledged=True)

In [13]:
## MongoDB 데이터 읽어오기
df = pd.DataFrame(
    list(train_collection.find())
)
df

,_id,temperature,pressure,vibration,quality_score
0,6a1d19bd7d6c50bc711d1cb8,70,100,0.10,80
1,6a1d19bd7d6c50bc711d1cb9,75,110,0.15,85
2,6a1d19bd7d6c50bc711d1cba,80,120,0.20,88
3,6a1d19bd7d6c50bc711d1cbb,85,130,0.25,92
4,6a1d19bd7d6c50bc711d1cbc,90,140,0.30,95


In [14]:
## 모델 학습
x = df[["temperature", "pressure", "vibration"]] # 온도, 압력, 진동
y = df.quality_score


## LinearRegression 모델 실행
model = LinearRegression()
model.fit(x,y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [16]:
## 신규 x 데이터에 대한 y 퀄리티 스코어 뽑는 분포함수 만들기 (Inference=추론)
new_data = pd.DataFrame(
    [
        {"temperature":82, "pressure":125, "vibration":0.22}, # 분석요청 데이터
    ]
)

## 추론(결과전송)
pred = model.predict(new_data)[0]
print(f"예측결과: {pred}")

예측결과: 89.7759940801184


In [17]:
## 결과 저장
predict_collection.insert_one(
        {"temperature":82, "pressure":125, "vibration":0.22, "predicted_quality": float(pred)},
)
print(f"예측결과: {pred}")

예측결과: 89.7759940801184
